# EDM U-Net: Memorization ↔ Generalization Transition

**Purpose.** Prof. Baptista asked to *"show a similar transition between memorization and
generalization for the U-Net"* (his §5.3–5.4 show it for MLPs/U-Nets via training time and
capacity: all models eventually memorize; early stopping / under-parameterization
regularize).

Setup follows the mechanism findings from `edm_unet_memorization_mechanism.ipynb`:
- **Matched sampler** (σ_max = 10, 1000 SDE steps — "config D" of the σ-fix study), since
  the σ_max = 80 sampler evaluates the net far outside its training range and hides
  memorization that is present in the weights.
- Axes of the transition: **training set size** n_train ∈ {2, 4, 8, 16, 32} × **training
  time** up to 30k steps with log-spaced checkpoints (Baptista Fig. 17–18: memorization
  emerges with enough optimization; their N=2 U-Net needed ~50k epochs).
- Metrics per checkpoint: per-band memorization ratio (coarse/fine; <1 = memorized), plus
  the Baptista-style pixel-space collapse measure (relative L2 distance of each generated
  sample to its nearest training field). GMM closed form = memorization ceiling.

**Expectation:** memorized at small n_train / late training, novel at large n_train / early
stopping — the U-Net analogue of the paper's transition, with the per-band metric showing
*which scales* memorize first.

**Runtime:** ~10–20 min per n_train for training (30k steps, MPS) + ~30 s per checkpoint
evaluation → order 1.5–2.5 h for the full grid. Set `SMOKE = True` for a minutes-long
end-to-end check. Checkpoints are saved incrementally, so the eval section can be rerun
without retraining.

In [ ]:
import sys, os, math, time, copy
import numpy as np
import torch
import matplotlib.pyplot as plt

# -- path setup --
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import diffusion_score_models as score_models
from multiband_data_utils import generate_multiband_dataset_postmask
from memorization_metrics import RingMetricContext
from edm import EDMPrecond, EDMScoreWrapper, train_edm
from unet import SmallUNet, count_parameters
from device_utils import resolve_device

DEVICE = resolve_device()   # cuda > mps > cpu; use resolve_device("cpu") on the personal laptop
print(f'device: {DEVICE}')

In [ ]:
# -- Data generation (identical config to the earlier memorization notebooks) --
components = [
    {"name": "coarse", "length_scale": 2.0,  "s": 2.0, "sigma_sq": 1.0, "band": (0.5, 4.0)},
    {"name": "mid1",   "length_scale": 6.0,  "s": 2.0, "sigma_sq": 1.0, "band": (4.0, 10.0)},
    {"name": "mid2",   "length_scale": 12.0, "s": 2.0, "sigma_sq": 1.0, "band": (10.0, 18.0)},
    {"name": "fine",   "length_scale": 24.0, "s": 2.0, "sigma_sq": 1.0, "band": (18.0, 32.0)},
]
result = generate_multiband_dataset_postmask(
    num_samples=200, grid_size=128, components=components,
    weights=[1.0, 0.8, 0.8, 1.2], seed=42, normalize=True,
)
bands = result.get('bands', {c['name']: c['band'] for c in components})
N = 128
x_all = result['combined']          # kept on CPU; slices moved to DEVICE as needed
ctx = RingMetricContext(N, bands, device=DEVICE)
print(f'x_all: {tuple(x_all.shape)}')

In [ ]:
# -- Evaluation: matched sampler (sigma_max=10, config D of the sigma-fix study) --
VE_SAMPLE = score_models.VE_EDM(sigma_min=0.002, sigma_max=10.0)
N_GEN = 16
N_SDE_STEPS = 1000
N_RAND_REF = 32
LATENT_SEED = 42

@torch.no_grad()
def sample_from(score_fn):
    torch.manual_seed(LATENT_SEED)   # same latents (and step noise stream) for every run
    latents = torch.randn(N_GEN, N*N, device=DEVICE)
    out = VE_SAMPLE.SDEsampler(score_fn, latents, num_steps=N_SDE_STEPS)
    return out.reshape(N_GEN, N, N)

@torch.no_grad()
def pixel_nn_stats(x_gen, x_train):
    '''Relative pixel-space L2 distance to the nearest training field (Baptista-style
    collapse measure). Returns per-sample distances; threshold at plot time.'''
    d = torch.cdist(x_gen.flatten(1), x_train.flatten(1))
    nn_rel = d.min(dim=1).values / x_train.flatten(1).norm(dim=1).mean()
    return nn_rel.cpu()

@torch.no_grad()
def evaluate_checkpoint(precond, x_train):
    wrapper = EDMScoreWrapper(precond, VE_SAMPLE.marginal_prob_std, N, c_tikhonov=0.0).to(DEVICE)
    x_gen = sample_from(wrapper)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'mean_ratio': m['mean_ratio'].cpu(),
        'nn_rel': pixel_nn_stats(x_gen, x_train),
        'samples': x_gen[:2].cpu(),
    }

@torch.no_grad()
def gmm_reference(x_train):
    train_flat = x_train.reshape(x_train.shape[0], -1)
    gmm = score_models.GMM_score(train_flat, VE_SAMPLE.marginal_prob_mean,
                                 VE_SAMPLE.marginal_prob_std)
    x_gen = sample_from(gmm)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'nn_rel': pixel_nn_stats(x_gen, x_train),
    }

results_dir = os.path.join(repo_root, 'results', 'data')
os.makedirs(results_dir, exist_ok=True)
fig_dir = os.path.join(repo_root, 'results', 'figures')
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
# -- Sweep configuration --
SMOKE = False    # True: tiny end-to-end run to verify the pipeline

N_TRAIN_SWEEP = [2, 4, 8, 16, 32]
TOTAL_STEPS = 30000
CHECKPOINT_AT = [250, 500, 1000, 2000, 4000, 8000, 16000, 30000]
BATCH_SIZE = 8

if SMOKE:
    N_TRAIN_SWEEP = [2, 4]
    TOTAL_STEPS = 200
    CHECKPOINT_AT = [100, 200]
    N_SDE_STEPS = 50
    N_GEN = 4

ckpt_path = os.path.join(results_dir, 'edm_unet_transition_checkpoints.pt')
print(f'n_train sweep: {N_TRAIN_SWEEP}, steps: {TOTAL_STEPS}, checkpoints: {CHECKPOINT_AT}')

In [ ]:
# -- Training (checkpoints saved incrementally per n_train) --
all_ckpts = {}
if os.path.exists(ckpt_path):
    all_ckpts = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'loaded existing checkpoints for n_train = {sorted(all_ckpts.keys())}')

for n_train in N_TRAIN_SWEEP:
    if n_train in all_ckpts:
        print(f'n_train={n_train}: already trained, skipping')
        continue
    print(f'===== training n_train = {n_train} =====')
    x_train = x_all[:n_train].to(DEVICE)
    train_flat = x_train.reshape(n_train, -1)
    t0 = time.time()
    saved = train_edm(train_flat, grid_size=N, total_steps=TOTAL_STEPS,
                      checkpoint_at=CHECKPOINT_AT, base_channels=16, emb_dim=64,
                      lr=1e-3, batch_size=BATCH_SIZE, seed=0, device=DEVICE,
                      UNetClass=SmallUNet)
    all_ckpts[n_train] = {
        step: {'state_dict': {k: v.cpu() for k, v in p.state_dict().items()},
               'sigma_data': p.sigma_data}
        for step, p in saved.items()
    }
    torch.save(all_ckpts, ckpt_path)
    print(f'  {time.time()-t0:.0f}s; saved -> {ckpt_path}')

In [ ]:
# -- Evaluation: memorization vs training step, per n_train + GMM ceiling --
eval_results = {}
gmm_refs = {}
for n_train in N_TRAIN_SWEEP:
    x_train = x_all[:n_train].to(DEVICE)
    print(f'===== evaluating n_train = {n_train} =====')
    gmm_refs[n_train] = gmm_reference(x_train)
    print(f"  GMM ceiling: coarse={gmm_refs[n_train]['coarse_score']:.4f} "
          f"fine={gmm_refs[n_train]['fine_score']:.4f}")
    eval_results[n_train] = {}
    for step, entry in sorted(all_ckpts[n_train].items()):
        unet = SmallUNet(base_channels=16, emb_dim=64).to(DEVICE)
        precond = EDMPrecond(unet, sigma_data=entry['sigma_data']).to(DEVICE)
        precond.load_state_dict(entry['state_dict'])
        precond.eval()
        t0 = time.time()
        r = evaluate_checkpoint(precond, x_train)
        eval_results[n_train][step] = r
        frac03 = (r['nn_rel'] < 0.3).float().mean().item()
        print(f"  step {step:>6}: coarse={r['coarse_score']:.4f} fine={r['fine_score']:.4f} "
              f"nn_rel(med)={r['nn_rel'].median():.3f} frac<0.3={frac03:.2f} ({time.time()-t0:.0f}s)")

torch.save({'eval_results': eval_results, 'gmm_refs': gmm_refs,
            'checkpoint_at': CHECKPOINT_AT, 'n_train_sweep': N_TRAIN_SWEEP,
            'sampler': {'sigma_max': 10.0, 'n_steps': N_SDE_STEPS, 'latent_seed': LATENT_SEED},
            'note': ('EDM SmallUNet (C=16,E=64,L=3) memorization vs training step per n_train; '
                     'matched sampler sigma_max=10, 1000 steps; ratio metric + pixel NN rel dist; '
                     'GMM closed form as memorization ceiling.')},
           os.path.join(results_dir, 'edm_unet_transition_results.pt'))
print('saved -> edm_unet_transition_results.pt')

In [ ]:
# -- Plots: the transition --
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
cmap = plt.cm.plasma
colors = {n: cmap(i / max(len(N_TRAIN_SWEEP)-1, 1)) for i, n in enumerate(N_TRAIN_SWEEP)}

for ax, key, title in [(axes[0], 'coarse_score', 'coarse band'),
                       (axes[1], 'fine_score', 'fine band')]:
    for n_train in N_TRAIN_SWEEP:
        steps = sorted(eval_results[n_train].keys())
        ys = [eval_results[n_train][s][key] for s in steps]
        ax.plot(steps, ys, marker='o', ms=4, color=colors[n_train], label=f'n={n_train}')
        ax.axhline(gmm_refs[n_train][key], color=colors[n_train], lw=0.8, ls=':')
    ax.axhline(1.0, color='gray', lw=0.8, ls='--')
    ax.set_xscale('log')
    ax.set_xlabel('training step')
    ax.set_title(f'{title} (dotted = GMM ceiling)')
axes[0].set_ylabel('band score (<1 = memorized)')
axes[0].legend(fontsize=8)

ax = axes[2]
for n_train in N_TRAIN_SWEEP:
    steps = sorted(eval_results[n_train].keys())
    ys = [(eval_results[n_train][s]['nn_rel'] < 0.3).float().mean().item() for s in steps]
    ax.plot(steps, ys, marker='o', ms=4, color=colors[n_train], label=f'n={n_train}')
ax.set_xscale('log')
ax.set_xlabel('training step')
ax.set_ylabel('fraction of samples with rel NN dist < 0.3')
ax.set_title('pixel-space collapse fraction')
fig.suptitle('U-Net memorization vs training time and n_train (matched sampler, sigma_max=10)', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'unet_transition_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## Notes

- If no configuration memorizes even at 30k steps, extend `TOTAL_STEPS` for the small
  n_train runs (Baptista's N=2 image U-Net needed ~50k epochs) before concluding the EDM
  U-Net resists memorization — that would itself be a finding worth reporting, given the
  mechanism notebook shows the memorized basins exist but are unreachable from pure noise.
- The GMM dotted lines are the memorization ceiling for the same sampler/latents; the
  interesting quantity is how far each U-Net curve descends toward its ceiling and at which
  scale (coarse first, per the earlier scale-selective results).
- Checkpoints live in `results/data/edm_unet_transition_checkpoints.pt` and are reused by
  the capacity sweep notebook workflow if needed.